# AeroPulse Anomaly — Phase 2: Shared Feature Layer

Turn the imported base table into the **feature store** that every downstream
anomaly notebook (03–09) reads. Implements sections 11–16 of the
*Anomaly Model Review & Improvement Plan*.

## Objective

Phase 2 previously produced lags, rolls and cyclical encodings. That was enough
for a residual model but, as the review concluded, not enough physical context
to separate *"PM2.5 is high"* from *"PM2.5 is unexpectedly high"*.

This version adds the four driver families the plan asks for:

| Plan section | Family | What it buys |
|---|---|---|
| 11 | Extended PM2.5 dynamics — lags to 168h, slopes, acceleration, 24h quantiles | Episode shape, not just level |
| 12 | Wind-relative fire transport — 6 radii × 4 decay windows, upwind/crosswind/downwind | Attribution of crop-burning and industrial plumes |
| 13 | Regional PM2.5 transport — 25/50/100 km neighbours, upwind neighbour mean | Delhi-NCR and Indo-Gangetic transport |
| 14 | CAMS satellite — AOD, NO₂, CO, SO₂, dust + anomalies | Column signal the ground network cannot see |
| 15 | Meteorological derivatives — rain windows, deltas, ventilation, stagnation | Dispersion physics |
| 16 | Static geography — elevation, relief, IGP membership | Distinguishes station regimes |

It also builds the **multi-target block** from section 17, so notebooks 05–08 can
train against a hazard label instead of thresholding a residual.

### What this notebook still refuses to do

- No network calls. Every driver is read from `pm25_estimator`'s persisted Parquet.
- No `baseline_pm25`, `residual` or `robust_z`. Those are fit on training rows
  only, after the split, inside notebooks 03–09.
- No future-valued feature. Every rolling and lag operation is backward-looking;
  the only forward-looking columns are the explicitly named `target_*` labels.

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


## 1. Load the Phase-1 base table

In [2]:
# ============================================================================
# LOAD PHASE-1 BASE
# ============================================================================
BASE_FILE = CFG["base_dir"] / "anomaly_base.parquet"
if not BASE_FILE.exists():
    raise FileNotFoundError(f"{BASE_FILE} not found. Run notebook 01 first.")

t0 = time.time()
df = pd.read_parquet(BASE_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

# Phase 2 of the pm25 estimator already wrote some of these under different
# definitions. Drop them so this notebook is the single source of truth and a
# re-run cannot silently mix two generations of the same feature.
# Anything this notebook regenerates is dropped rather than merged. Two columns
# sharing a label makes df["pm25_lag_1h"] return a DataFrame, and every
# downstream arithmetic on it fails in a way that is hard to trace back here.
REGENERATED_PREFIXES = (
    "pm25_lag_", "pm25_roll_", "pm25_delta_", "pm25_slope_", "pm25_accel_",
    "pm25_pct_change_", "pm25_z_", "pm25_vs_", "pm25_range_",
    "fire_count_", "fire_frp_", "fire_min_dist", "fire_min_distance",
    "fire_distance_", "fire_invdist", "fire_near_far", "fire_upwind",
    "fire_crosswind", "fire_downwind", "fire_active",
    "neighbour_pm25", "national_pm25", "national_station", "regional_pm25",
    "upwind_pm25", "cams_", "rain_", "delta_temp", "delta_pressure",
    "delta_pblh", "delta_wind", "delta_rh", "sin_", "cos_", "target_",
)
REGENERATED_EXACT = {
    "hour", "dow", "hour_of_week", "doy", "month", "is_weekend", "season",
    "pm25_local_excess", "ventilation_proxy", "ventilation_coefficient",
    "inv_ventilation", "stagnation_flag", "stagnation_index",
    "low_wind_flag", "low_pbl_flag", "satellite_available",
    "elevation_m", "region_key", "geo_block",
}
SUPERSEDED = [c for c in df.columns
              if c.startswith(REGENERATED_PREFIXES) or c in REGENERATED_EXACT]
df = df.drop(columns=SUPERSEDED)
tk.assert_unique_columns(df)
print(f"loaded {len(df):,} rows / {df['location_id'].nunique()} stations "
      f"in {time.time()-t0:.1f}s")
print(f"dropped {len(SUPERSEDED)} superseded upstream columns")
print("span:", df["timestamp_utc"].min(), "->", df["timestamp_utc"].max())

loaded 1,705,252 rows / 149 stations in 0.3s
dropped 69 superseded upstream columns
span: 2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00


### Result — imported dataset

Expect ~1.7M rows across 149 stations spanning Feb 2025 → Sep 2026. The
superseded columns are dropped rather than kept alongside the new ones: the
upstream `fire_count_50km` counted *all* detections in a fixed window, while
this notebook produces radius × decay-window × wind-sector variants that carry
the same prefix.

## 2. Time keys and cyclical encodings

`hour_of_week` is the grouping **key** for every baseline in notebooks 03–09,
not a model input. Sine/cosine pairs keep hour 23 adjacent to hour 0 and
December adjacent to January.

In [3]:
# ============================================================================
# TIME FEATURES
# ============================================================================
ts = df["timestamp_utc"]
df["hour"] = ts.dt.hour.astype("int16")
df["dow"] = ts.dt.dayofweek.astype("int16")
df["hour_of_week"] = (df["dow"] * 24 + df["hour"]).astype("int16")
df["doy"] = ts.dt.dayofyear.astype("int16")
df["month"] = ts.dt.month.astype("int16")
df["is_weekend"] = (df["dow"] >= 5).astype("int8")
df["season"] = tk.season_of(df["month"])

for name, period, series in (
    ("hour", 24, df["hour"]), ("doy", 366, df["doy"]), ("how", 168, df["hour_of_week"])
):
    df[f"sin_{name}"] = np.sin(2 * np.pi * series / period).astype("float32")
    df[f"cos_{name}"] = np.cos(2 * np.pi * series / period).astype("float32")

print("hour_of_week:", int(df.hour_of_week.min()), "-", int(df.hour_of_week.max()))
print(df["season"].value_counts().to_string())

hour_of_week: 0 - 167


season
monsoon         672215
pre_monsoon     562242
winter          280954
post_monsoon    189841


## 3. Extended PM2.5 temporal dynamics (plan section 11)

The previous feature set stopped at a 48h lag. Pollution episodes in the
Indo-Gangetic Plain routinely run longer than that, so weekly context
(`lag_168h`, `roll_mean_168h`) is what lets a model tell *"this station is
always dirty"* from *"this station has been dirty since Tuesday"*.

Every rolling window is `shift(1)` before aggregating: the value at hour *t*
describes hours strictly before *t*, so a feature can never contain its own row.

In [4]:
# ============================================================================
# PM2.5 DYNAMICS — all backward-looking
# shift(1) before rolling is the leakage guard: without it the window at hour t
# includes pm25(t), which is the thing every downstream model is predicting.
# ============================================================================
LAGS = (1, 2, 3, 6, 12, 24, 48, 72, 168)
ROLL_WINDOWS = (3, 6, 12, 24, 48, 72, 168)

g = df.groupby("location_id", sort=False)["pm25"]
prev = g.shift(1)                      # everything below is built from this
gp = prev.groupby(df["location_id"], sort=False)

t0 = time.time()
new = {}
for h in LAGS:
    new[f"pm25_lag_{h}h"] = g.shift(h)

for w in ROLL_WINDOWS:
    r = gp.rolling(w, min_periods=max(2, w // 4))
    new[f"pm25_roll_mean_{w}h"] = r.mean().reset_index(level=0, drop=True)
    if w >= 6:
        new[f"pm25_roll_std_{w}h"] = r.std().reset_index(level=0, drop=True)

r24 = gp.rolling(24, min_periods=6)
new["pm25_roll_min_24h"] = r24.min().reset_index(level=0, drop=True)
new["pm25_roll_max_24h"] = r24.max().reset_index(level=0, drop=True)
new["pm25_roll_p90_24h"] = r24.quantile(0.90).reset_index(level=0, drop=True)
new["pm25_roll_p95_24h"] = r24.quantile(0.95).reset_index(level=0, drop=True)

df = tk.attach(df, pd.DataFrame(new, index=df.index).astype("float32"))
tk.assert_unique_columns(df)
print(f"lags + rolls in {time.time()-t0:.1f}s")

lags + rolls in 1.9s


In [5]:
# ============================================================================
# DELTAS, SLOPES, ACCELERATION
# Rate of change is what separates an episode ONSET from an episode PLATEAU.
# A detector that only sees the level cannot tell those apart.
# ============================================================================
dyn = {}
for h in (1, 3, 6, 12, 24):
    dyn[f"pm25_delta_{h}h"] = df["pm25"] - df[f"pm25_lag_{h}h"]
    dyn[f"pm25_slope_{h}h"] = dyn[f"pm25_delta_{h}h"] / h

for h in (1, 6, 24):
    denom = df[f"pm25_lag_{h}h"].replace(0, np.nan)
    dyn[f"pm25_pct_change_{h}h"] = (df["pm25"] - df[f"pm25_lag_{h}h"]) / denom

# Acceleration: is the rise itself speeding up?
dyn["pm25_accel_3h"] = dyn["pm25_slope_3h"] - (
    (df["pm25_lag_3h"] - df["pm25_lag_6h"]) / 3.0
)
dyn["pm25_accel_6h"] = dyn["pm25_slope_6h"] - (
    (df["pm25_lag_6h"] - df["pm25_lag_12h"]) / 6.0
)

# Where does the current reading sit inside its own recent distribution?
spread = df["pm25_roll_std_24h"].replace(0, np.nan)
dyn["pm25_z_vs_24h"] = (df["pm25"] - df["pm25_roll_mean_24h"]) / spread
dyn["pm25_vs_roll_168h"] = df["pm25"] - df["pm25_roll_mean_168h"]
dyn["pm25_range_24h"] = df["pm25_roll_max_24h"] - df["pm25_roll_min_24h"]

df = tk.attach(df, pd.DataFrame(dyn, index=df.index).astype("float32"))
print("dynamics columns:", len(dyn))

dynamics columns: 18


### Result — dynamics block

`pm25_z_vs_24h` and `pm25_accel_*` are the columns that matter most for the
onset model in notebook 06. They are deliberately *self-referential to the
station's own recent history* rather than to a fitted baseline, so they can be
computed here without leaking the train/test cut.

## 4. Meteorological derivatives (plan section 15)

Raw temperature and wind speed are weak predictors on their own. The
dispersion-relevant quantities are the **ventilation coefficient**
(`wind_speed × PBLH`, how much air volume is available to dilute an emission)
and the **stagnation index** (low wind *and* shallow boundary layer *and* no
rain), which is the classic Delhi winter smog setup.

In [6]:
# ============================================================================
# WEATHER DERIVATIVES
# ============================================================================
gl = df.groupby("location_id", sort=False)
met = {}

if "precipitation" in df.columns:
    pr = gl["precipitation"].shift(1).groupby(df["location_id"], sort=False)
    for w in (1, 3, 6, 12, 24, 48):
        met[f"rain_{w}h"] = pr.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)

for col, short in (("temperature_2m", "temp"), ("surface_pressure", "pressure"),
                   ("boundary_layer_height", "pblh"), ("wind_speed_10m", "wind"),
                   ("relative_humidity_2m", "rh")):
    if col in df.columns:
        for h in (3, 24):
            met[f"delta_{short}_{h}h"] = df[col] - gl[col].shift(h)

ws = df.get("wind_speed_10m")
pbl = df.get("boundary_layer_height")
if ws is not None and pbl is not None:
    met["ventilation_coefficient"] = ws * pbl
    met["inv_ventilation"] = 1.0 / (ws * pbl + 1.0)
    pbl_low = pbl.quantile(0.25)
    met["low_pbl_flag"] = (pbl < pbl_low).astype("float32")
    met["low_wind_flag"] = (ws < 1.5).astype("float32")
    rain24 = met.get("rain_24h", pd.Series(0.0, index=df.index))
    met["stagnation_index"] = (
        (ws < 1.5).astype(float) + (pbl < pbl_low).astype(float) + (rain24 < 0.1).astype(float)
    )
    # Interactions the plan calls out: dispersion state x current pollution.
    met["lowpbl_x_pm25lag1"] = (pbl < pbl_low).astype(float) * df["pm25_lag_1h"]
    met["lowwind_x_pm25lag1"] = (ws < 1.5).astype(float) * df["pm25_lag_1h"]
    if "relative_humidity_2m" in df.columns:
        met["rh_x_pm25lag1"] = df["relative_humidity_2m"] * df["pm25_lag_1h"] / 100.0

df = tk.attach(df, pd.DataFrame(met, index=df.index).astype("float32"))
print("weather derivative columns:", len(met))
print(df[["ventilation_coefficient", "stagnation_index"]].describe().T.to_string())

weather derivative columns: 24
                             count         mean          std  min    25%     50%     75%       max
ventilation_coefficient  1705252.0  6734.413086  9388.187500  0.0  675.0  3088.5  9223.5  144820.0
stagnation_index         1705252.0     0.786934     0.789134  0.0    0.0     1.0     1.0       3.0


## 5. Wind-relative fire transport (plan section 12)

The upstream fire features counted detections inside fixed radii. Proximity
alone is the wrong question: a 500 MW fire 80 km **downwind** is irrelevant,
while the same fire **upwind** will be over the station within hours.

The raw FIRMS detections carry `fire_lat` / `fire_lon` / `frp`, so we can
compute the bearing from each station to each fire and compare it against
`wind_direction_10m` (the meteorological convention — the direction wind blows
*from*). A fire whose bearing matches the wind direction is upwind.

Doing this row-by-row would be a 1.7M × 1.7M cross product. Instead the toolkit
collapses detections onto a dense `(station, hour, sector)` cube once, then
projects it onto rows — the trailing sums become cumulative-sum slices, so
window length costs nothing.

In [7]:
# ============================================================================
# FIRE TRANSPORT CUBES
# ============================================================================
FIRE_FILE = PM25_BASE / "fire_detections.parquet"
if not FIRE_FILE.exists():
    raise FileNotFoundError(f"{FIRE_FILE} missing — run pm25_estimator Phase 1.")

fires = pd.read_parquet(FIRE_FILE)
fires["timestamp_utc"] = pd.to_datetime(fires["timestamp_utc"], utc=True)
print(f"{len(fires):,} FIRMS detections, "
      f"{fires['timestamp_utc'].min().date()} -> {fires['timestamp_utc'].max().date()}")

grid = tk.HourGrid.build(df["timestamp_utc"], df["location_id"])
stations = (df.groupby("location_id")[["lat", "lon"]].first()
              .reindex(grid.station_ids).reset_index())
print(f"grid: {grid.n_stations} stations x {grid.n_hours:,} hours")

t0 = time.time()
fire_cubes = tk.build_fire_transport_cubes(fires, stations, grid)
print(f"fire cubes built in {time.time()-t0:.1f}s")

1,692,885 FIRMS detections, 2025-02-18 -> 2026-09-09
grid: 149 stations x 13,621 hours


fire cubes built in 5.1s


In [8]:
# ============================================================================
# PROJECT FIRE CUBES ONTO ROWS (wind-relative)
# ============================================================================
row_si = grid.station_index(df["location_id"])
row_hi = grid.hour_index(df["timestamp_utc"])

t0 = time.time()
fire_feats = tk.fire_features_for_rows(
    fire_cubes, grid, row_si, row_hi,
    df["wind_direction_10m"].to_numpy("float64"),
)
fire_feats.index = df.index
df = tk.attach(df, fire_feats)
del fire_cubes
print(f"{fire_feats.shape[1]} fire features in {time.time()-t0:.1f}s")

_up = [c for c in fire_feats.columns if "upwind_frp" in c and "150km" in c]
print(df[_up].describe().T[["mean", "50%", "max"]].to_string())

134 fire features in 4.8s
                                mean   50%           max
fire_upwind_frp_150km_6h   11.666303  0.00   9687.970703
fire_upwind_frp_150km_12h  23.569424  0.00   9687.970703
fire_upwind_frp_150km_24h  48.809200  1.09  10222.081055
fire_upwind_frp_150km_48h  97.096130  5.34  16873.320312


### Result — fire transport

`fire_upwind_frp_150km_24h` is the headline column: total fire radiative power
in the 90° arc the wind is arriving from, over the previous 24 hours. Its
median is 0 (most station-hours have no upwind fire at all) with a long right
tail during the Punjab/Haryana burning windows — which is exactly the sparse,
seasonal, high-signal shape a transport feature should have.

`fire_upwind_share_*` is NaN wherever no fire was detected in that radius,
rather than 0. That distinction matters: "no fires anywhere" and "fires present
but all downwind" are different states, and collapsing both to zero would throw
away the signal.

## 6. Regional PM2.5 transport (plan section 13)

Local PM2.5 cannot explain Delhi-NCR episodes driven by upwind Punjab or the
wider Indo-Gangetic Plain. These features give each station its neighbours'
**concurrent** readings, resolved by radius and by wind direction.

Concurrent is legitimate: at prediction time hour *t* the network has already
reported hour *t* for every other station. Nothing here reaches into the future.

In [9]:
# ============================================================================
# REGIONAL / UPWIND PM2.5
# ============================================================================
t0 = time.time()
regional = tk.build_regional_pm25_features(df, stations, grid)
df = tk.attach(df, regional)
print(f"{regional.shape[1]} regional features in {time.time()-t0:.1f}s")

# National context — the "is the whole country having a bad day" signal.
#
# Computed LEAVE-ONE-OUT. A plain groupby mean over all stations at hour t
# includes this station's own pm25(t), and national_pm25_max is simply equal to
# it whenever the station is the dirtiest in the country — which is exactly the
# subpopulation a hazard model cares about. Subtracting the row's own
# contribution keeps the feature purely external.
grp = df.groupby("timestamp_utc")["pm25"]
own = df["pm25"].to_numpy("float64")
tot = grp.transform("sum").to_numpy("float64")
cnt = grp.transform("count").to_numpy("float64")
sq = grp.transform(lambda s: (s ** 2).sum()).to_numpy("float64")

n_other = np.maximum(cnt - 1.0, 0.0)
with np.errstate(invalid="ignore", divide="ignore"):
    loo_mean = np.where(n_other > 0, (tot - own) / n_other, np.nan)
    loo_var = np.where(n_other > 0, (sq - own ** 2) / n_other - loo_mean ** 2, np.nan)

# Leave-one-out max needs the runner-up: if this station holds the maximum, the
# external maximum is the second highest reading that hour.
rank = grp.rank(method="first", ascending=False)
top1 = grp.transform("max").to_numpy("float64")
second = df["pm25"].where(rank == 2).groupby(df["timestamp_utc"]).transform("max").to_numpy("float64")
loo_max = np.where(rank.to_numpy() == 1, second, top1)

df = tk.attach(df, pd.DataFrame({
    "national_pm25_mean": loo_mean,
    "national_pm25_std": np.sqrt(np.clip(loo_var, 0.0, None)),
    "national_pm25_max": loo_max,
    "national_station_count": n_other,
}, index=df.index).astype("float32"))
df["pm25_vs_national"] = (df["pm25"] - df["national_pm25_mean"]).astype("float32")
print("national aggregates are leave-one-out (own station excluded)")

print(df[["regional_pm25_mean_100km", "upwind_pm25_mean",
          "regional_pm25_gradient_50km"]].describe().T.to_string())

18 regional features in 0.6s


national aggregates are leave-one-out (own station excluded)
                                 count       mean        std          min        25%        50%        75%          max
regional_pm25_mean_100km     1439176.0  36.941292  28.841360     0.000000  18.666666  29.500000  46.900002  1520.000000
upwind_pm25_mean              968645.0  36.214504  31.755033     0.000000  17.100000  27.510000  46.000000  1450.000000
regional_pm25_gradient_50km  1125938.0  -0.130556  30.681667 -1845.199951  -9.206054  -0.803333   8.100001  1845.199951


> **Leakage note.** `pm25_vs_national` and `regional_pm25_gradient_*` contain
> the row's own `pm25`. They are safe for the **+24h hazard** models in
> notebooks 05–08, where the target is a strictly future quantity, but they are
> *not* safe for a nowcast of `pm25(t)`. Notebook 03 excludes them explicitly.

## 7. CAMS satellite drivers (plan section 14)

`satellite_hourly.parquet` carries CAMS reanalysis co-located to each station:
AOD, NO₂, CO, SO₂, dust and O₃. AOD is column-integrated aerosol — it sees the
plume aloft before it mixes down to the surface monitor, which is precisely the
lead-time signal a 24h hazard model needs.

The plan warns against blindly forward-filling satellite data, so the join is
exact on `(location_id, hour)` and gaps stay NaN. LightGBM handles NaN natively.

In [10]:
# ============================================================================
# SATELLITE JOIN
# ============================================================================
SAT_FILE = PM25_BASE / "satellite_hourly.parquet"
if SAT_FILE.exists():
    sat = pd.read_parquet(SAT_FILE)
    sat["timestamp_utc"] = pd.to_datetime(sat["timestamp_utc"], utc=True)

    # The CAMS grid was already sampled at the station coordinates upstream, so
    # each grid point maps to exactly one station by rounded lat/lon.
    key = lambda f: (f["lat"].round(3).astype(str) + "_" + f["lon"].round(3).astype(str))
    sat["_k"] = key(sat)
    loc_key = stations.assign(_k=key(stations))[["location_id", "_k"]]
    sat = sat.merge(loc_key, on="_k", how="inner").drop(columns=["_k", "lat", "lon"])
    print(f"satellite matched to {sat['location_id'].nunique()} / {len(stations)} stations")

    SAT_COLS = [c for c in sat.columns if c.startswith("cams_")]
    sat = sat.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
    gs = sat.groupby("location_id", sort=False)
    add = {}
    for c in ("cams_aod", "cams_no2", "cams_pm25"):
        if c in sat.columns:
            prev_c = gs[c].shift(1).groupby(sat["location_id"], sort=False)
            add[f"{c}_change_3h"] = sat[c] - gs[c].shift(3)
            add[f"{c}_roll_24h"] = prev_c.rolling(24, min_periods=6).mean().reset_index(level=0, drop=True)
            add[f"{c}_roll_168h"] = prev_c.rolling(168, min_periods=24).mean().reset_index(level=0, drop=True)
    sat = tk.attach(sat, pd.DataFrame(add, index=sat.index).astype("float32"))
    # Anomaly = today's column load against this station's own weekly norm.
    for c in ("cams_aod", "cams_no2", "cams_pm25"):
        if f"{c}_roll_168h" in sat.columns:
            sat[f"{c}_anomaly"] = (sat[c] - sat[f"{c}_roll_168h"]).astype("float32")

    # CAMS is stamped on the hour; ground observations land at :30. Join on the
    # floored hour rather than the raw timestamp, which otherwise matches nothing.
    sat = sat.rename(columns={"timestamp_utc": "_hour"})
    sat["_hour"] = sat["_hour"].dt.floor("h")
    df["_hour"] = df["timestamp_utc"].dt.floor("h")
    before = df.shape[1]
    df = df.merge(sat, on=["location_id", "_hour"], how="left").drop(columns=["_hour"])
    df["satellite_available"] = df["cams_aod"].notna().astype("int8")
    print(f"satellite columns added: {df.shape[1] - before + 1}")
    print("satellite coverage:", round(df["satellite_available"].mean(), 4))
    if df["satellite_available"].mean() < 0.5:
        print("WARNING: satellite coverage below 50% — check the station/lat-lon key.")
    print(df[[c for c in ("cams_aod", "cams_no2", "cams_co")
              if c in df.columns]].describe().T.to_string())
else:
    print("WARNING: no satellite file; AOD/NO2 features will be absent.")
    df["satellite_available"] = 0

satellite matched to 136 / 149 stations


satellite columns added: 22
satellite coverage: 0.9108
              count        mean         std    min     25%     50%     75%          max
cams_aod  1553196.0    0.511654    0.287893   0.03    0.31    0.46    0.65     4.080000
cams_no2  1553196.0   13.150360   12.734211   0.00    4.50    9.30   17.50   220.100006
cams_co   1553196.0  356.525146  235.337616  58.00  201.00  296.00  437.00  4984.000000


## 8. Static geography (plan section 16)

`static_geography.parquet` supplies elevation, local relief and Indo-Gangetic
Plain membership. The plan also asks for population, road and building density
and land use — **those are not in the current dataset** and are flagged as a
known gap rather than silently approximated.

In [11]:
# ============================================================================
# STATIC GEOGRAPHY + REGION KEYS
# ============================================================================
GEO_FILE = PM25_BASE / "static_geography.parquet"
geo = pd.read_parquet(GEO_FILE)
geo = tk.build_region_key(geo)          # adds region_key + geo_block
GEO_COLS = ["location_id", "elevation_m", "elevation_std_10km", "elevation_range_10km",
            "elevation_relief_10km", "is_indo_gangetic_plain", "region_key", "geo_block"]
df = df.merge(geo[[c for c in GEO_COLS if c in geo.columns]], on="location_id", how="left")

geo.to_parquet(CFG["feature_dir"] / "station_geometry.parquet", index=False)
print("regions:", geo["region_key"].nunique(), "| geo blocks:", geo["geo_block"].nunique())
print("IGP stations:", int(geo["is_indo_gangetic_plain"].sum()), "/", len(geo))
print("MISSING (plan s.16, not in dataset): population / road / building density, land use")

regions: 50 | geo blocks: 48
IGP stations: 67 / 149
MISSING (plan s.16, not in dataset): population / road / building density, land use


In [12]:
# ============================================================================
# CALENDAR DRIVERS — Diwali and crop-burning windows
# ============================================================================
CAL_FILE = PM25_BASE / "calendar_events.parquet"
if CAL_FILE.exists():
    cal = pd.read_parquet(CAL_FILE)
    cal["date"] = pd.to_datetime(cal["date"]).dt.date
    keep = [c for c in cal.columns if c != "holiday_name"]
    df["_date"] = df["timestamp_utc"].dt.date
    df = df.merge(cal[keep], left_on="_date", right_on="date", how="left").drop(
        columns=["_date", "date"], errors="ignore")
    for c in ("diwali_window", "festival_window", "crop_burning_window",
              "crop_burning_kharif", "crop_burning_rabi", "is_holiday"):
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int8")
    print("calendar columns merged:", [c for c in keep if c != "date"])
else:
    print("no calendar_events.parquet — skipping")

calendar columns merged: ['days_to_diwali', 'days_after_diwali', 'diwali_window', 'festival_window', 'days_to_nearest_festival', 'is_holiday', 'crop_burning_kharif', 'crop_burning_rabi', 'crop_burning_window']


## 9. Multi-target block (plan section 17)

This is the change that matters most. The old pipeline had one implicit label —
*is the current hour above 121?* — reachable only by thresholding a residual.

Section 9 of the plan splits the problem into three objectives, so Phase 2 now
emits the targets all three need:

- **Model A (forecast)** — `target_pm25_t_plus_{1,3,6,12,24,48}h`
- **Model B (hazard)** — `target_{poor,very_poor,severe}_within_24h`, built from
  `target_peak_pm25_next_24h`
- **Model C (onset)** — `target_onset_next_6h`, a *rise* rather than a *level*

Every target is computed with `shift(-h)` inside a station group and is the
only forward-looking content in the table. Downstream notebooks must never pass
a `target_*` column as a feature.

In [13]:
# ============================================================================
# TARGETS — the ONLY forward-looking columns in this table
# ============================================================================
gt = df.groupby("location_id", sort=False)["pm25"]
tgt = {}
for h in (1, 3, 6, 12, 24, 48):
    tgt[f"target_pm25_t_plus_{h}h"] = gt.shift(-h)

# Forward peak: reverse the series, take a trailing max, reverse back — this is
# max over (t, t+H], strictly excluding the current hour.
def forward_peak(hours):
    rev = df.iloc[::-1]
    out = (rev.groupby("location_id", sort=False)["pm25"]
              .shift(1).groupby(rev["location_id"], sort=False)
              .rolling(hours, min_periods=1).max().reset_index(level=0, drop=True))
    return out.iloc[::-1]

for h in (6, 12, 24, 48):
    tgt[f"target_peak_pm25_next_{h}h"] = forward_peak(h)

df = pd.concat([df, pd.DataFrame(tgt, index=df.index).astype("float32")], axis=1)

# Hazard labels at the three CPCB breakpoints that drive public messaging.
H = CFG["horizon_hours"]
peak = df[f"target_peak_pm25_next_{H}h"]
df[f"target_poor_within_{H}h"] = (peak >= CFG["poor_ugm3"]).astype("int8").where(peak.notna())
df[f"target_very_poor_within_{H}h"] = (peak >= CFG["very_poor_ugm3"]).astype("int8").where(peak.notna())
df[f"target_severe_within_{H}h"] = (peak >= CFG["severe_ugm3"]).astype("int8").where(peak.notna())

# Model C: an ONSET is a sharp rise relative to the station's own recent spread,
# regardless of absolute level. This is the "is this unusual?" question.
rise6 = df.groupby("location_id", sort=False)["pm25"].shift(-6) - df["pm25"]
spread = df["pm25_roll_std_24h"].replace(0, np.nan)
df["target_rise_next_6h"] = rise6.astype("float32")
df["target_onset_next_6h"] = ((rise6 / spread) >= 3.0).astype("int8").where(
    rise6.notna() & spread.notna())

print("target prevalence")
for c in [c for c in df.columns if c.startswith("target_") and df[c].dropna().isin([0, 1]).all()]:
    s = df[c].dropna()
    print(f"  {c:<34} {s.mean():.4%}  (n={len(s):,})")

target prevalence


  target_poor_within_24h             20.1759%  (n=1,705,103)
  target_very_poor_within_24h        11.2467%  (n=1,705,103)
  target_severe_within_24h           2.1386%  (n=1,705,103)
  target_onset_next_6h               5.2957%  (n=1,694,379)


### Result — the label problem, quantified

This is the single most important output of the rebuild. Compare prevalence:

| Label | Prevalence | Positives |
|---|---:|---:|
| Old: `pm25(t) >= 121` on the monsoon test slice | **0.37%** | 951 |
| New: `peak PM2.5 in next 24h >= 121` (full table) | **~11%** | ~190,000 |

The plan's section 19 argued the old detector was fighting extreme imbalance.
It was — but a large part of that imbalance was **self-inflicted by the label
definition**, not intrinsic to the problem. Asking *"will any hour in the next
day be hazardous?"* instead of *"is this specific hour hazardous?"* is a ~30×
increase in positive density, and it is also the question an early-warning
product actually needs to answer.

No amount of LightGBM tuning against the old label would have recovered that.

## 10. Leakage audit and persist

Before writing, assert the invariants the review demanded. A silent
reintroduction of `baseline_pm25` or a target-derived flag is the failure mode
that previously drove the detector's excess score to zero.

In [14]:
# ============================================================================
# LEAKAGE AUDIT
# ============================================================================
BANNED = ["baseline_pm25", "residual", "robust_z", "pm25_local_excess",
          "pm25_severe_flag", "pm25_extreme_flag"]
found = [c for c in BANNED if c in df.columns]
if found:
    print("dropping banned columns (computed on the full table / derived from target):", found)
    df = df.drop(columns=found)

# Any target must be genuinely forward-looking: correlation with the CURRENT
# pm25 should be well below 1.0, and the column must not equal a shifted self.
audit = {}
for c in [c for c in df.columns if c.startswith("target_pm25_t_plus")]:
    audit[c] = float(df[[c, "pm25"]].corr().iloc[0, 1])
print("\ntarget vs current pm25 correlation (must be < 1.0):")
for k, v in audit.items():
    flag = "LEAK" if v > 0.999 else "ok"
    print(f"  {k:<32} {v:.4f}  {flag}")
assert all(v < 0.999 for v in audit.values()), "a target is identical to a feature"

# Concurrent-pm25 features: safe for +24h models, unsafe for nowcasting pm25(t).
CONCURRENT_PM25 = [c for c in df.columns if c in
                   ("pm25_vs_national", "pm25_z_vs_24h", "pm25_vs_roll_168h",
                    "pm25_range_24h") or c.startswith("regional_pm25_gradient")
                   or c.startswith("upwind_pm25_gradient")]
print("\nconcurrent-pm25 features (exclude when nowcasting pm25(t)):")
print(" ", CONCURRENT_PM25)

dropping banned columns (computed on the full table / derived from target): ['pm25_severe_flag', 'pm25_extreme_flag']

target vs current pm25 correlation (must be < 1.0):
  target_pm25_t_plus_1h            0.8418  ok
  target_pm25_t_plus_3h            0.6589  ok
  target_pm25_t_plus_6h            0.5386  ok
  target_pm25_t_plus_12h           0.4965  ok
  target_pm25_t_plus_24h           0.5660  ok
  target_pm25_t_plus_48h           0.4950  ok

concurrent-pm25 features (exclude when nowcasting pm25(t)):
  ['pm25_z_vs_24h', 'pm25_vs_roll_168h', 'pm25_range_24h', 'regional_pm25_gradient_25km', 'regional_pm25_gradient_50km', 'regional_pm25_gradient_100km', 'upwind_pm25_gradient', 'pm25_vs_national']


In [15]:
# ============================================================================
# PERSIST FEATURE STORE + MANIFEST
# ============================================================================
ID_COLS = ["timestamp_utc", "location_id", "sensor_id", "station_name", "lat", "lon",
           "pm25", "pm25_unit", "source", "weather_source", "satellite_source",
           "source_quality", "schema_version", "timestamp_local", "grid_id",
           "season", "region_key", "geo_block", "pollution_regime"]

# Upstream event aggregates describe the WHOLE episode a row belongs to, peak
# included, so at hour t they already contain hours after t. They are retained
# for the episode analysis in notebook 06 but must never be model inputs.
LEAKY_EVENT_COLS = [c for c in df.columns if c.startswith("event_")]

TARGET_COLS = [c for c in df.columns if c.startswith("target_")]
FEATURE_COLS = [c for c in df.columns
                if c not in ID_COLS and c not in TARGET_COLS
                and c not in CONCURRENT_PM25 and c not in LEAKY_EVENT_COLS]
print("event aggregate columns quarantined as leaky:", LEAKY_EVENT_COLS)

# float32 across the board: 1.7M rows x ~250 columns is 1.6 GB in float64.
tk.assert_unique_columns(df)
num = df.select_dtypes(include=["float64"]).columns
df[num] = df[num].astype("float32")

out = CFG["feature_dir"] / "anomaly_features.parquet"
df.to_parquet(out, index=False)

manifest = {
    "schema_version": CFG["schema_version"],
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "source": "anomaly_phase1_base + pm25 drivers (fire / satellite / geography / calendar)",
    "rows": int(len(df)),
    "stations": int(df["location_id"].nunique()),
    "window_start_utc": str(df["timestamp_utc"].min()),
    "window_end_utc": str(df["timestamp_utc"].max()),
    "n_columns": int(df.shape[1]),
    "feature_candidates": FEATURE_COLS,
    "target_columns": TARGET_COLS,
    "concurrent_pm25_features": CONCURRENT_PM25,
    "leaky_event_columns": LEAKY_EVENT_COLS,
    "id_columns": [c for c in ID_COLS if c in df.columns],
    "baseline_deferred": True,
    "fingerprint": tk.dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]]),
    "plan_sections_implemented": [11, 12, 13, 14, 15, 16, 17],
    "known_gaps": [
        "No population / road / building density or land use (plan s.16).",
        "19 months only — one winter. Plan s.28 asks for 3-5 years.",
        "Weather is reanalysis actuals, not a forecast. A true +24h operational "
        "model needs forecast weather at inference (plan s.30).",
    ],
    "notes": [
        "hour_of_week is a baseline KEY produced here; the baseline itself is not.",
        "baseline_pm25 / residual / robust_z are intentionally absent.",
        "target_* are the only forward-looking columns; never pass them as features.",
        "concurrent_pm25_features embed pm25(t) — valid for +24h targets only.",
        "event_* aggregates span the whole episode and are never model inputs.",
    ],
}
tk.write_report(CFG["feature_dir"] / "dataset_manifest.json", manifest)

size_mb = out.stat().st_size / 1e6
print(f"wrote {out}  ({size_mb:.0f} MB)")
print(f"rows={len(df):,}  columns={df.shape[1]}  features={len(FEATURE_COLS)}  targets={len(TARGET_COLS)}")

event aggregate columns quarantined as leaky: ['event_start_flag', 'event_id', 'event_start', 'event_end', 'event_peak_pm25', 'event_mean_pm25', 'event_observed_hours', 'event_duration_hours', 'event_gap_hours']


wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/data/anomaly/processed/features/anomaly_features.parquet  (512 MB)
rows=1,705,252  columns=325  features=274  targets=15


### Result — feature store

The shared feature layer is now on disk and notebooks 03–09 consume it without
recomputing anything. Rough family sizes:

| Family | Approx. columns |
|---|---:|
| PM2.5 dynamics (lags, rolls, slopes, quantiles) | ~45 |
| Fire transport (radii × decay × wind sector) | ~134 |
| Regional / upwind PM2.5 | ~20 |
| CAMS satellite + anomalies | ~20 |
| Weather derivatives | ~25 |
| Time / calendar / geography | ~25 |

**Three gaps are recorded in the manifest rather than papered over:**

1. **No land-use or population density.** Plan section 16 asks for these to
   separate traffic-dominated from industrial from rural stations; the dataset
   does not carry them.
2. **19 months, one winter.** Plan section 28 asks for 3–5 years. This bounds
   how much the rolling-origin evaluation in notebook 09 can actually prove
   about winter generalisation — there is one winter, so a winter fold can be
   validated but never *replicated*.
3. **Weather is reanalysis, not forecast.** Every model here sees observed
   weather at time *t* only, which is correct. But a deployed +24h hazard model
   would want forecast weather over the horizon, and swapping analysis for
   forecast at serving time is a train/serve skew the plan flags in section 30.